Convert input embeddings into key,query, value vectors

In [3]:
import torch

inputs_posttrained = torch.tensor([ [0.44, 0.15, 0.89], #Your
                                    [0.55, 0.87, 0.66], #journey
                                    [0.53, 0.85, 0.67], #starts
                                    [0.22, 0.58, 0.33], #with
                                    [0.77, 0.25, 0.10], #one
                                    [0.05, 0.80, 0.55]])  #step



In [8]:
# for the word journey
x_2 = inputs_posttrained[1]

# fixed size of all the weight matrices (query, key. value) will be 3 x 2
dim_in = inputs_posttrained.shape[1]

dim_out = 2

In [18]:
torch.manual_seed(42)

W_query = torch.nn.Parameter(torch.rand(dim_in,dim_out) , requires_grad=False)

W_key = torch.nn.Parameter(torch.rand(dim_in,dim_out) , requires_grad=False)

W_value = torch.nn.Parameter(torch.rand(dim_in,dim_out) , requires_grad=False)

print("Weight matrix -> Query:", W_query )
      
print("\n Weight matrix -> Key:", W_key)

print("\n Weight matrix -> Value:", W_value)


Weight matrix -> Query: Parameter containing:
tensor([[0.8823, 0.9150],
        [0.3829, 0.9593],
        [0.3904, 0.6009]])

 Weight matrix -> Key: Parameter containing:
tensor([[0.2566, 0.7936],
        [0.9408, 0.1332],
        [0.9346, 0.5936]])

 Weight matrix -> Value: Parameter containing:
tensor([[0.8694, 0.5677],
        [0.7411, 0.4294],
        [0.8854, 0.5739]])


In [16]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print("Query vector (journey): ", query_2)
print("Key vector (journey): ", key_2)
print("Value vector (journey): ", value_2)

Query vector (journey):  tensor([1.0760, 1.7344])
Key vector (journey):  tensor([1.5764, 0.9441])
Value vector (journey):  tensor([1.7073, 1.0646])


Now, to obtain all query, key, and values

In [20]:
queries = inputs_posttrained @ W_query
keys = inputs_posttrained @ W_key
values = inputs_posttrained @ W_value

print(queries.shape)
print(keys.shape)
print(values.shape)

torch.Size([6, 2])
torch.Size([6, 2])
torch.Size([6, 2])


In [35]:
attention_scores = queries @ keys.T

attention_scores = attention_scores/ (keys.shape[-1] ** 0.5)

attention_weights = torch.softmax(attention_scores, dim=-1)

context_vectors = attention_weights @ values

print(attention_scores)

print("\n", attention_weights)

print(torch.sum((attention_weights[2])))

tensor([[1.2951, 1.6060, 1.5882, 0.8530, 0.8332, 1.0790],
        [1.9268, 2.3574, 2.3308, 1.2419, 1.2635, 1.5533],
        [1.8904, 2.3125, 2.2864, 1.2182, 1.2399, 1.5234],
        [1.0251, 1.2457, 1.2316, 0.6536, 0.6785, 0.8126],
        [1.2625, 1.5781, 1.5607, 0.8422, 0.8028, 1.0723],
        [1.1597, 1.3935, 1.3775, 0.7259, 0.7794, 0.8936]])

 tensor([[0.1729, 0.2359, 0.2318, 0.1111, 0.1089, 0.1393],
        [0.1741, 0.2678, 0.2608, 0.0878, 0.0897, 0.1198],
        [0.1743, 0.2659, 0.2590, 0.0890, 0.0910, 0.1208],
        [0.1760, 0.2195, 0.2164, 0.1214, 0.1245, 0.1423],
        [0.1714, 0.2350, 0.2310, 0.1126, 0.1082, 0.1417],
        [0.1784, 0.2254, 0.2218, 0.1156, 0.1220, 0.1367]])
tensor(1.0000)


Dividing by square root to reduce variance and make learning stable

Query: Represents current token that the model is focusing on

Key: In attention mechanism, every item in input sequence has a key. Keys are used to match with the query

Value: Represents actual content of input items. Once model determines which keys are most relevant to the query, it retrieves that corresponding value